# CH03_07. 완전한 리서치 에이전트 구현

이번 실습에서는 지금까지 배운 모든 기술을 통합하여 **완전한 리서치 에이전트**를 구현합니다.

## 🎯 학습 목표
1. **통합 아키텍처**: TODO, 로컬 파일 시스템, 서브 에이전트를 모두 통합
2. **검색 도구**: 웹 검색 결과를 실제 파일로 저장하고 요약만 반환
3. **think_tool**: 전략적 사고를 위한 성찰 도구
4. **컨텍스트 오프로딩**: 긴 결과는 파일에, 요약만 컨텍스트에 유지

## 📚 아키텍처 개요

```
┌─────────────────────────────────────────────────────────┐
│                    메인 에이전트                         │
│  - TODO 관리 (write_todos, read_todos)                  │
│  - 파일 관리 (ls, read_file, write_file) → 로컬 디스크  │
│  - 전략적 사고 (think_tool)                              │
│  - 서브 에이전트 위임 (task)                             │
└─────────────────────┬───────────────────────────────────┘
                      │
              task 도구 호출
                      │
                      ▼
┌─────────────────────────────────────────────────────────┐
│               리서치 서브 에이전트                        │
│  - 웹 검색 (web_search) → 파일 자동 저장                │
│  - 전략적 사고 (think_tool)                              │
│  [격리된 컨텍스트에서 실행]                               │
└─────────────────────────────────────────────────────────┘
        │
        ▼
  📁 agent_workspace/
     ├── search_langchain_2026-02-04.md
     ├── search_langgraph_2026-02-04.md
     └── comparison_report.md
```

## `create_agent` vs `create_deep_agent` 비교

이 노트북에서는 `create_agent`를 사용해 TODO, 파일 시스템, 서브 에이전트를 **직접 구현**합니다. 하지만 LangChain의 [`deepagents`](https://github.com/langchain-ai/deepagents) 라이브러리를 사용하면 이 모든 것을 **한 줄**로 해결할 수 있습니다.

### 핵심 차이점

| 기능 | `create_agent` (수동 구현) | `create_deep_agent` (내장) |
|------|----------------------------|---------------------------|
| **TODO 관리** | `write_todos`, `read_todos` 직접 구현 | 내장 (`write_todos` 자동 제공) |
| **파일 시스템** | `ls`, `read_file`, `write_file` 직접 구현 | 내장 (`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`) |
| **서브 에이전트** | `SubAgent` 타입 + `_create_task_tool` 직접 구현 | 내장 (`task` 도구 + `subagents` 파라미터) |
| **컨텍스트 관리** | 수동으로 오프로딩 로직 구현 | 자동 요약 + 대용량 출력 처리 |
| **시스템 프롬프트** | 모든 도구 사용법을 직접 작성 | Claude Code 스타일 기본 프롬프트 내장 |
| **파일 백엔드** | 상태 기반 또는 로컬 디스크 직접 구현 | `StateBackend`, `FilesystemBackend`, `StoreBackend` 선택 |
| **셸 실행** | 미지원 | `execute` 도구 내장 (샌드박싱) |
| **장기 기억** | 미지원 | `AGENTS.md` + LangGraph Store 연동 |

### 왜 둘 다 알아야 하나?

- **`create_agent` (수동)**: 내부 동작 원리를 이해하고, 세밀한 커스터마이징이 필요할 때
- **`create_deep_agent` (내장)**: 프로덕션 수준의 에이전트를 빠르게 구축할 때

> 이 노트북의 섹션 2~9에서는 수동 구현을 통해 원리를 학습하고,
> **마지막 섹션**에서 `create_deep_agent`로 같은 기능을 간단하게 구현하는 방법을 보여줍니다.

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv
import warnings

load_dotenv()
warnings.filterwarnings("ignore")

# 환경 변수 확인
print("=== LangSmith 설정 상태 ===")
print(f"LANGSMITH_API_KEY: {'설정됨' if os.getenv('LANGSMITH_API_KEY') else '미설정 ⚠️'}")
print(f"LANGSMITH_TRACING: {os.getenv('LANGSMITH_TRACING', '미설정')}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT', '미설정 (default 사용)')}")
print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정 ⚠️'}")

# 프로젝트 이름 설정 (선택)
os.environ["LANGSMITH_PROJECT"] = "fc-agent-bible-appendix"
print("fc-agent-bible-appendix 프로젝트의 Tracing이 활성화되었습니다.")

In [ ]:
from pathlib import Path
from datetime import datetime

# 이미 최상단에서 환경변수 및 경고 설정이 끝났으므로 여기서는 중복 제거
# (os, warnings, dotenv 관련 코드는 생략)

# EXA API 클라이언트 초기화
from exa_py import Exa
exa_client = Exa(api_key=os.environ["EXA_API_KEY"])

# 🗂️ 워크스페이스 디렉토리 설정
WORKSPACE_DIR = Path("./agent_workspace")
WORKSPACE_DIR.mkdir(exist_ok=True)

def get_today_str() -> str:
    return datetime.now().strftime("%Y-%m-%d")

print(f"✅ 환경 설정 완료")
print(f"📁 워크스페이스: {WORKSPACE_DIR.absolute()}")
print(f"🔍 EXA API 클라이언트 초기화 완료")

## 2. 상태 및 기본 도구 정의

In [ ]:
from typing_extensions import TypedDict, Annotated, Optional, NotRequired
from langgraph.graph.message import add_messages
from langchain_core.messages import ToolMessage, HumanMessage
from langchain_core.tools import tool, InjectedToolCallId, BaseTool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from IPython.display import Image, display

# Todo 정의
class Todo(TypedDict):
    content: str
    status: str

# 에이전트 상태 정의 (파일은 로컬 디스크에 저장)
class DeepAgentState(TypedDict):
    """DeepAgent의 완전한 상태 스키마
    
    Note: 파일은 WORKSPACE_DIR에 저장되므로 상태에 포함하지 않음
    """
    messages: Annotated[list, add_messages]
    todos: NotRequired[list[Todo]]

print("✅ 상태 스키마가 정의되었습니다.")

## 3. TODO 관리 도구

In [ ]:
WRITE_TODOS_DESCRIPTION = """
복잡한 워크플로우를 추적하기 위한 구조화된 작업 목록을 생성합니다.

## 언제 사용
- 여러 단계가 필요한 복잡한 작업
- 사용자가 여러 작업을 요청했을 때

## 구조
- 상태는 pending, in_progress, completed 중 하나
- 한 번에 하나의 in_progress 작업만
"""

@tool(description=WRITE_TODOS_DESCRIPTION)
def write_todos(
    todos: list[Todo],
    tool_call_id: Annotated[str, InjectedToolCallId]
) -> Command:
    """TODO 목록을 업데이트합니다."""
    return Command(
        update={
            "todos": todos,
            "messages": [
                ToolMessage(f"TODO 업데이트됨: {todos}", tool_call_id=tool_call_id)
            ],
        }
    )

@tool
def read_todos(
    state: Annotated[DeepAgentState, InjectedState],
) -> str:
    """현재 TODO 목록을 읽습니다."""
    todos = state.get("todos", [])
    if not todos:
        return "TODO가 없습니다."
    
    result = "📋 현재 TODO:\n"
    for i, todo in enumerate(todos, 1):
        emoji = {"pending": "⏳", "in_progress": "🔄", "completed": "✅"}.get(todo["status"], "❓")
        result += f"{i}. {emoji} {todo['content']} ({todo['status']})\n"
    return result.strip()

print("✅ TODO 도구 정의 완료")

## 4. 파일 관리 도구

In [ ]:
@tool
def ls(path: str = ".") -> str:
    """워크스페이스 디렉토리의 파일을 나열합니다."""
    target_dir = WORKSPACE_DIR / path
    
    if not target_dir.exists():
        return f"오류: 경로 '{path}'가 존재하지 않습니다."
    
    items = []
    for item in sorted(target_dir.iterdir()):
        if item.is_dir():
            items.append(f"📁 {item.name}/")
        else:
            size = item.stat().st_size
            items.append(f"📄 {item.name} ({size} bytes)")
    
    return "\n".join(items) if items else "(파일이 없습니다)"

@tool
def read_file(file_path: str, offset: int = 0, limit: int = 2000) -> str:
    """워크스페이스에서 파일 내용을 읽습니다."""
    target_file = WORKSPACE_DIR / file_path
    
    # 보안: 워크스페이스 외부 접근 방지
    try:
        target_file = target_file.resolve()
        if not str(target_file).startswith(str(WORKSPACE_DIR.resolve())):
            return "오류: 워크스페이스 외부 접근 불가"
    except Exception:
        return f"오류: 잘못된 경로 '{file_path}'"
    
    if not target_file.exists():
        return f"오류: '{file_path}' 파일 없음"
    
    try:
        content = target_file.read_text(encoding="utf-8")
        lines = content.splitlines()
        end_idx = min(offset + limit, len(lines))
        result_lines = [f"{i+1:6d}\t{lines[i][:2000]}" for i in range(offset, end_idx)]
        return "\n".join(result_lines)
    except Exception as e:
        return f"오류: 파일 읽기 실패 - {e}"

@tool
def write_file(file_path: str, content: str) -> str:
    """워크스페이스에 파일을 작성합니다."""
    target_file = WORKSPACE_DIR / file_path
    
    try:
        target_file.parent.mkdir(parents=True, exist_ok=True)
        target_file = target_file.resolve()
        if not str(target_file).startswith(str(WORKSPACE_DIR.resolve())):
            return "오류: 워크스페이스 외부 접근 불가"
        
        target_file.write_text(content, encoding="utf-8")
        return f"✅ '{file_path}' 저장됨 ({target_file.stat().st_size} bytes)"
    except Exception as e:
        return f"오류: 파일 쓰기 실패 - {e}"

print("✅ 파일 도구 정의 완료 (로컬 디스크 사용)")

## 5. 전략적 사고 도구 (think_tool)

`think_tool`은 에이전트가 리서치 진행 상황을 성찰하고 다음 단계를 계획하는 데 사용됩니다.

In [ ]:
@tool
def think_tool(reflection: str) -> str:
    """리서치 진행 상황에 대한 전략적 성찰 도구.
    
    각 검색 후에 이 도구를 사용하여 결과를 분석하고 다음 단계를 체계적으로 계획하세요.
    
    성찰 시 다루어야 할 내용:
    1. 현재 발견한 정보 분석 - 어떤 구체적인 정보를 수집했는가?
    2. 갭 평가 - 아직 부족한 핵심 정보는 무엇인가?
    3. 품질 평가 - 좋은 답변을 위한 충분한 근거/예시가 있는가?
    4. 전략적 결정 - 계속 검색할 것인가, 답변을 제공할 것인가?
    
    Args:
        reflection: 리서치 진행, 발견, 갭, 다음 단계에 대한 상세한 성찰
    
    Returns:
        의사결정을 위해 성찰이 기록되었음을 확인
    """
    return f"💭 성찰 기록됨: {reflection[:200]}..."

print("✅ think_tool 정의 완료")

## 6. 웹 검색 도구 (컨텍스트 오프로딩)

검색 결과의 전체 내용은 파일에 저장하고, 요약만 반환하여 컨텍스트를 효율적으로 관리합니다.

In [ ]:
# EXA 기반 웹 검색 도구 (컨텍스트 오프로딩 적용)
@tool(parse_docstring=True)
def web_search(query: str) -> str:
    """Search the web for information using EXA and save results to a file.
    
    This tool performs semantic web search using EXA API. Full results are
    saved to a local file for context offloading, and a concise summary is returned.
    
    Context Offloading Pattern:
    - Full search results → saved to local file
    - Concise summary → returned to agent
    
    Args:
        query: The search query - be specific and clear about what you're looking for
    
    Returns:
        A summary of search results with the filename where full content is saved
    """
    try:
        # EXA 검색 수행
        results = exa_client.search_and_contents(
            query=query,
            num_results=5,
            text={"max_characters": 2000},
            highlights={"num_sentences": 3}
        )
        
        # 파일에 저장할 상세 내용 구성
        file_content = f"# Search Results: {query}\n"
        file_content += f"**Date:** {get_today_str()}\n"
        file_content += f"**Query:** {query}\n\n"
        file_content += "---\n\n"
        
        for i, result in enumerate(results.results, 1):
            file_content += f"## {i}. {result.title}\n"
            file_content += f"**URL:** {result.url}\n\n"
            
            if result.highlights:
                file_content += "### Key Highlights:\n"
                for highlight in result.highlights:
                    file_content += f"- {highlight}\n"
            
            if result.text:
                file_content += f"\n### Full Text:\n{result.text}\n"
            
            file_content += "\n---\n\n"
        
        # 파일명 생성 및 저장
        safe_query = query.lower().replace(" ", "_")[:30]
        filename = f"search_{safe_query}_{get_today_str()}.md"
        target_file = WORKSPACE_DIR / filename
        target_file.write_text(file_content, encoding="utf-8")
        
        # 컨텍스트에 반환할 요약
        summary = f"""🔍 Search completed: "{query}"

📄 Full results saved to: {filename} ({target_file.stat().st_size} bytes)

📝 Key findings ({len(results.results)} results):
"""
        for i, result in enumerate(results.results[:3], 1):
            summary += f"{i}. **{result.title}**\n"
            if result.highlights:
                summary += f"   → {result.highlights[0][:150]}...\n"
        
        summary += f"\n💡 Use read_file('{filename}') to see full details."
        
        return summary
        
    except Exception as e:
        return f"Search error: {str(e)}"

print("✅ EXA 웹 검색 도구 정의 완료 (로컬 파일 자동 저장)")

## 7. 서브 에이전트 및 task 도구 설정

In [ ]:
class SubAgent(TypedDict):
    name: str
    description: str
    prompt: str
    tools: NotRequired[list[str]]

# task 도구 설명 템플릿
TASK_DESCRIPTION_PREFIX = """
특화된 서브 에이전트에게 작업을 위임합니다.
각 서브 에이전트는 격리된 컨텍스트에서 실행됩니다.

## 사용 가능한 서브 에이전트:
{other_agents}
"""

def create_task_tool(tools, subagents: list[SubAgent], model, state_schema):
    """task 도구를 생성합니다."""
    agents = {}
    tools_by_name = {t.name: t for t in tools if isinstance(t, BaseTool)}
    
    for _agent in subagents:
        _tools = [tools_by_name[t] for t in _agent.get("tools", [])] if "tools" in _agent else tools
        agents[_agent["name"]] = create_agent(
            model, tools=_tools, system_prompt=_agent["prompt"], state_schema=state_schema
        )
    
    other_agents_string = "\n".join([f"- **{a['name']}**: {a['description']}" for a in subagents])
    
    @tool(description=TASK_DESCRIPTION_PREFIX.format(other_agents=other_agents_string))
    def task(
        description: str,
        subagent_type: str,
    ) -> str:
        """서브 에이전트에 작업 위임
        
        Note: 파일은 로컬 디스크에 저장되므로 서브 에이전트가 
        저장한 파일은 메인 에이전트에서도 접근 가능
        """
        if subagent_type not in agents:
            return f"오류: '{subagent_type}' 없음. 사용 가능: {list(agents.keys())}"
        
        sub_agent = agents[subagent_type]
        
        # 격리된 컨텍스트 (메시지만 새로 시작)
        isolated_state = {"messages": [{"role": "user", "content": description}]}
        
        result = sub_agent.invoke(isolated_state)
        
        # 서브 에이전트의 마지막 응답 반환
        return result["messages"][-1].content
    
    return task

print("✅ task 도구 생성 함수 정의 완료")

## 8. 완전한 에이전트 조립

In [ ]:
# 모델 초기화
model = init_chat_model(model="openai:gpt-4o-mini", temperature=0.0)

# 리서치 서브 에이전트 프롬프트 (레퍼런스: RESEARCHER_INSTRUCTIONS)
RESEARCHER_INSTRUCTIONS = """You are a research assistant conducting research on the user's input topic. For context, today's date is {date}.

<Task>
Your job is to use tools to gather information about the user's input topic.
You can use any of the tools provided to you to find resources that can help answer the research question.
</Task>

<Available Tools>
1. **web_search**: For conducting web searches to gather information
   - Results are automatically saved to files for context offloading
   - A summary is returned to you
2. **think_tool**: For reflection and strategic planning during research
3. **read_file**: To read full details from saved search results
4. **write_file**: To save your analysis or summaries

**CRITICAL: Use think_tool after each search to reflect on results and plan next steps**
</Available Tools>

<Instructions>
Think like a human researcher with limited time:
1. **Read the question carefully** - What specific information does the user need?
2. **Start with broader searches** - Use broad, comprehensive queries first
3. **After each search, pause and assess** - Do I have enough to answer? What's still missing?
4. **Execute narrower searches as needed** - Fill in the gaps
5. **Stop when you can answer confidently** - Don't keep searching for perfection
</Instructions>

<Hard Limits>
**Tool Call Budgets** (Prevent excessive searching):
- **Simple queries**: Use 1-2 search tool calls maximum
- **Normal queries**: Use 2-3 search tool calls maximum
- **Very Complex queries**: Use up to 5 search tool calls maximum

**Stop Immediately When**:
- You can answer the user's question comprehensively
- You have 3+ relevant examples/sources for the question
- Your last 2 searches returned similar information
</Hard Limits>
""".format(date=get_today_str())

research_sub_agent = {
    "name": "research-agent",
    "description": "Delegate research to the sub-agent researcher. Provide clear, specific research questions.",
    "prompt": RESEARCHER_INSTRUCTIONS,
    "tools": ["web_search", "write_file", "read_file", "think_tool"],
}

# 모든 도구
all_tools = [web_search, write_file, read_file, ls, think_tool]

# task 도구 생성
task_tool = create_task_tool(
    tools=all_tools,
    subagents=[research_sub_agent],
    model=model,
    state_schema=DeepAgentState,
)

# 메인 에이전트 프롬프트 (레퍼런스: SUBAGENT_USAGE_INSTRUCTIONS + TODO_USAGE_INSTRUCTIONS)
MAIN_AGENT_INSTRUCTIONS = """You are a research project manager coordinating research tasks. Today's date is {date}.

<Task>
Your role is to:
1. Analyze user requests and create a TODO plan
2. Coordinate research by delegating to sub-agents
3. Synthesize findings into final deliverables
</Task>

<Available Tools>
1. **write_todos / read_todos**: Manage your task list
2. **task(description, subagent_type)**: Delegate research to sub-agents
   - description: Clear, specific research question
   - subagent_type: "research-agent"
3. **ls, read_file, write_file**: Manage files in workspace
4. **think_tool**: Reflect on progress and plan next steps

**PARALLEL RESEARCH**: When you identify multiple independent research directions, make multiple **task** tool calls in a single response to enable parallel execution.
</Available Tools>

<Workflow>
1. **Plan**: Use write_todos to create research plan based on user request
2. **Research**: Use task to delegate each research topic to sub-agents
3. **Collect**: Use ls and read_file to review gathered information
4. **Synthesize**: Write final report using write_file
5. **Complete**: Mark all todos as completed
</Workflow>

<Hard Limits>
- **Stop when adequate** - Don't over-research; stop when you have sufficient information
- **Maximum 3 parallel task calls** per iteration
- **Maximum 5 total task delegations** for any request
</Hard Limits>

<Scaling Rules>
**Simple fact-finding**: Use 1 sub-agent

**Comparisons** (e.g., "Compare A vs B"):
- Use a sub-agent for each element
- Example: "Compare LangChain vs LangGraph" → 2 sub-agents

**Multi-faceted research**:
- Use parallel agents for different aspects
- Example: "Research AI agents: architecture, tools, use cases" → 3 sub-agents
</Scaling Rules>
""".format(date=get_today_str())

# 메인 에이전트 도구
main_agent_tools = [write_todos, read_todos, task_tool, ls, read_file, write_file, think_tool]

# 메인 에이전트 생성
main_agent = create_agent(
    model,
    tools=main_agent_tools,
    system_prompt=MAIN_AGENT_INSTRUCTIONS,
    state_schema=DeepAgentState,
)

print("✅ 완전한 리서치 에이전트가 생성되었습니다.")
print(f"🔍 EXA 웹 검색 활성화")
print("\n📊 에이전트 그래프 구조:")
display(Image(main_agent.get_graph(xray=True).draw_mermaid_png()))

## 9. 에이전트 실행

In [ ]:
# 에이전트 실행 (스트리밍 모드)
print("🚀 메인 에이전트 실행 시작 (스트리밍)")
print("=" * 60)

query = {
    "messages": [
        {
            "role": "user",
            "content": """LangChain과 LangGraph에 대해 각각 조사하고,
두 기술의 차이점과 활용 사례를 정리해서 'comparison_report.md' 파일로 저장해줘.""",
        }
    ],
    "todos": [],
}

# 최종 상태 저장용
final_state = None

# 스트리밍으로 실행 과정 확인 (서브그래프 포함)
for namespace, mode, chunk in main_agent.stream(
    query, 
    stream_mode=["updates", "values"], 
    subgraphs=True
):
    if mode == "updates":
        # 현재 그래프 이름 표시
        graph_name = namespace if len(namespace) > 0 else "main"
        
        for node_name, value in chunk.items():
            print(f"\n📍 [{graph_name}] → [{node_name}]")
            
            if "messages" in value and value["messages"]:
                last_msg = value["messages"][-1]
                
                # Tool Call 표시
                if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                    for tc in last_msg.tool_calls:
                        print(f"   🔧 Tool: {tc['name']}")
                        if tc.get('args'):
                            args_str = str(tc['args'])[:80]
                            print(f"   📝 Args: {args_str}...")
                
                # 응답 표시
                elif hasattr(last_msg, 'content') and last_msg.content:
                    content = str(last_msg.content)
                    if len(content) > 300:
                        print(f"   💬 {content[:300]}...")
                    else:
                        print(f"   💬 {content}")
    
    elif mode == "values":
        # 최신 상태 저장
        final_state = chunk

# 최종 결과 저장
result = final_state

print("\n" + "=" * 60)
print("📝 메인 에이전트 최종 응답:")
print("=" * 60)
if result and "messages" in result:
    print(result["messages"][-1].content)

## 10. 결과 확인

In [ ]:
# TODO 상태 확인
print("📋 최종 TODO 상태:")
print("=" * 70)
for i, todo in enumerate(result.get("todos", []), 1):
    emoji = {"pending": "⏳", "in_progress": "🔄", "completed": "✅"}.get(todo["status"], "❓")
    print(f"{i}. {emoji} {todo['content']} ({todo['status']})")

# 🔥 실제 디스크의 파일 목록 확인
print("\n📁 워크스페이스 파일 목록 (로컬 디스크):")
print("=" * 70)
print(f"📂 {WORKSPACE_DIR.absolute()}\n")

for item in sorted(WORKSPACE_DIR.iterdir()):
    if item.is_file():
        size = item.stat().st_size
        print(f"  📄 {item.name} ({size} bytes)")

In [ ]:
# 🔥 실제 디스크에서 파일 내용 확인
print("📄 저장된 파일 내용 (로컬 디스크에서 읽기):")
print("=" * 70)

for filepath in sorted(WORKSPACE_DIR.iterdir()):
    if filepath.is_file() and filepath.suffix == ".md":
        print(f"\n### {filepath.name} ###")
        print("-" * 40)
        content = filepath.read_text(encoding="utf-8")
        preview = content[:800] + "..." if len(content) > 800 else content
        print(preview)

## 📚 핵심 정리

### 완전한 리서치 에이전트의 구성 요소:

1. **TODO 관리**
   - `write_todos`: 작업 계획 작성
   - `read_todos`: 현재 진행 상황 확인

2. **로컬 파일 시스템** 🔥
   - `ls`: 워크스페이스 파일 목록
   - `read_file`: 실제 파일 읽기
   - `write_file`: 실제 파일 저장
   - 모든 파일은 `./agent_workspace/`에 저장됨

3. **EXA 웹 검색 + 컨텍스트 오프로딩** 🔥
   - EXA API로 실제 웹 검색
   - 검색 결과 → 로컬 파일에 자동 저장
   - 요약만 → 에이전트 컨텍스트에 반환

4. **서브 에이전트 위임**
   - `task`: 격리된 컨텍스트에서 서브 에이전트 실행
   - 병렬 리서치 지원
   - 파일은 공유되므로 서브 에이전트가 저장한 파일도 접근 가능

5. **전략적 사고**
   - `think_tool`: 진행 상황 성찰 및 다음 단계 계획

### XML 구조 프롬프트 패턴:

```xml
<Task>
Your role is to coordinate research by delegating to sub-agents.
</Task>

<Available Tools>
1. **web_search**: Search with auto file saving
2. **task**: Delegate to sub-agents
3. **think_tool**: Reflect on progress
</Available Tools>

<Hard Limits>
- Maximum search calls per query type
- Stop when adequate information gathered
- Limit total task delegations
</Hard Limits>

<Scaling Rules>
- Simple questions → 1 sub-agent
- Comparisons → 1 sub-agent per element
- Multi-faceted → parallel agents for each aspect
</Scaling Rules>
```

### EXA 웹 검색 + 컨텍스트 오프로딩:

```python
from exa_py import Exa
exa_client = Exa(api_key=os.environ["EXA_API_KEY"])

@tool
def web_search(query: str) -> str:
    # 1. EXA로 실제 웹 검색
    results = exa_client.search_and_contents(
        query=query,
        num_results=5,
        text={"max_characters": 2000},
        highlights={"num_sentences": 3}
    )
    
    # 2. 상세 결과는 파일에 저장
    target_file.write_text(file_content)
    
    # 3. 요약만 컨텍스트에 반환
    return f"검색 완료. 파일: {filename}"
```

### 핵심 패턴:
- **InjectedState**: 도구에서 상태 접근
- **Command**: 상태 업데이트 반환 (TODO용)
- **로컬 파일 시스템**: 실제 디스크에 파일 저장
- **EXA API**: 의미 기반 실제 웹 검색
- **create_agent**: 최신 LangChain 1.0 에이전트 생성

### 축하합니다! 🎉
이제 프로덕션 수준의 리서치 에이전트 아키텍처를 이해하셨습니다!

## 11. `create_deep_agent`로 간단하게 구현하기

위에서 TODO, 파일 시스템, 서브 에이전트, 검색 도구를 모두 **직접 구현**했습니다.
[`deepagents`](https://github.com/langchain-ai/deepagents) 라이브러리를 사용하면 이 모든 것이 **내장**되어 있어 훨씬 간결하게 동일한 기능을 구현할 수 있습니다.

### `create_deep_agent`의 내장 기능

`create_deep_agent()`를 호출하면 다음 도구들이 **자동으로** 포함됩니다:

| 카테고리 | 내장 도구 | 우리가 구현한 것 |
|----------|-----------|-----------------|
| **계획** | `write_todos`, `read_todos` | 섹션 3에서 직접 구현 |
| **파일** | `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep` | 섹션 4에서 직접 구현 |
| **서브 에이전트** | `task` (자동 컨텍스트 격리) | 섹션 7에서 직접 구현 |
| **셸** | `execute` (샌드박싱) | 미구현 |
| **컨텍스트** | 자동 요약, 대용량 출력 처리 | 수동 오프로딩 |

In [ ]:
# deepagents 설치
!pip install deepagents

In [ ]:
import os
from typing import Literal
from exa_py import Exa  # EXA Search 공급사로 변경
from deepagents import create_deep_agent

exa_client = Exa(api_key=os.environ["EXA_API_KEY"])

# 커스텀 도구 - EXA 웹 검색
def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """웹에서 정보를 검색합니다.

    주어진 쿼리에 대해 EXA semantic search를 수행합니다.
    최대 결과 수(max_results), 토픽(topic), 원본포함여부(include_raw_content)는 무시되거나 검색 쿼리 최적화에 참조하지만 EXA API-features로 제한됩니다.
    """
    # EXA는 topic, include_raw_content는 기본적으로 지원하지 않으나, 인터페이스 호환용 인자 유지
    results = exa_client.search_and_contents(
        query=query,
        num_results=max_results,
        text={"max_characters": 2000},
        highlights={"num_sentences": 3},
    )
    # 결과 요약만 반환(파일저장 등의 오프로드 로직은 기존과 동일하게 deepagents 내장 도구로 처리)
    highlights = []
    for i, result in enumerate(results.results[:3], 1):
        row = f"{i}. {result.title}"
        if result.highlights:
            row += f" - {result.highlights[0][:100]}..."
        highlights.append(row)
    return "검색결과 요약:\n" + "\n".join(highlights)

# 리서치 에이전트 프롬프트
research_instructions = """당신은 전문 리서처입니다. 철저한 조사를 수행한 후 완성도 높은 보고서를 작성하는 것이 당신의 역할입니다.

인터넷 검색 도구를 주요 정보 수집 수단으로 사용할 수 있습니다.

## `internet_search`

주어진 쿼리에 대해 인터넷 검색을 수행합니다. 최대 결과 수, 주제, 원본 콘텐츠 포함 여부를 지정할 수 있습니다.
"""

# create_deep_agent로 에이전트 생성 - 이것만으로 TODO, 파일, 서브 에이전트 등이 모두 내장됨!
deep_agent = create_deep_agent(
    tools=[internet_search],
    system_prompt=research_instructions,
)

print("create_deep_agent로 생성 완료!")
print("내장 도구: write_todos, read_todos, ls, read_file, write_file, edit_file, task 등")

In [ ]:
# create_deep_agent 실행
result = deep_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "LangChain과 LangGraph에 대해 각각 조사하고, 두 기술의 차이점과 활용 사례를 정리해서 comparison_report.md 파일로 저장해줘.",
            }
        ]
    }
)

print(result["messages"][-1].content)

### 비교: 수동 구현 vs `create_deep_agent`

위에서 수동으로 구현한 코드(섹션 2~8)와 `create_deep_agent` 코드를 비교해봅시다:

**수동 구현 (섹션 2~8):**
```python
# 상태 정의
class DeepAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    todos: NotRequired[list[Todo]]

# TODO 도구 직접 구현
@tool
def write_todos(todos, tool_call_id): ...

@tool
def read_todos(state): ...

# 파일 도구 직접 구현
@tool
def ls(path): ...

@tool
def read_file(file_path, offset, limit): ...

@tool
def write_file(file_path, content): ...

# 서브 에이전트 도구 직접 구현
class SubAgent(TypedDict): ...
def create_task_tool(tools, subagents, model, state_schema): ...

# 프롬프트 직접 작성 (수십 줄)
MAIN_AGENT_INSTRUCTIONS = """..."""
RESEARCHER_INSTRUCTIONS = """..."""

# 에이전트 조립
main_agent = create_agent(model, tools=[...], system_prompt=..., state_schema=...)
```

**`create_deep_agent` (위 코드):**
```python
from deepagents import create_deep_agent

deep_agent = create_deep_agent(
    tools=[internet_search],
    system_prompt="당신은 전문 리서처입니다..."
)
```

> TODO, 파일 시스템, 서브 에이전트, 컨텍스트 관리가 모두 **자동으로** 포함됩니다.
> 커스텀 도구(`internet_search`)와 시스템 프롬프트만 전달하면 됩니다.

### `create_deep_agent` 주요 파라미터

```python
create_deep_agent(
    model="openai:gpt-4o",              # 모델 (기본: claude-sonnet-4-5)
    tools=[internet_search],             # 커스텀 도구
    system_prompt="...",                 # 시스템 프롬프트 (기본 프롬프트에 추가)
    subagents=[{                         # 서브 에이전트 설정 (선택)
        "name": "research-agent",
        "description": "리서치 위임",
        "prompt": "...",
        "tools": ["internet_search"],
    }],
    backend=FilesystemBackend(           # 파일 백엔드 (선택)
        root_dir="./agent_workspace"
    ),
    checkpointer=MemorySaver(),          # 체크포인터 (선택)
    memory=["./AGENTS.md"],              # 장기 기억 (선택)
)
```

### 파일 백엔드 옵션

| 백엔드 | 설명 | 사용 사례 |
|--------|------|-----------|
| **StateBackend** (기본) | LangGraph 상태에 저장 | 임시 스크래치 패드 |
| **FilesystemBackend** | 로컬 디스크에 저장 | 개발 CLI, CI/CD |
| **StoreBackend** | LangGraph Store에 저장 | 크로스 스레드 영구 저장 |